In [ ]:
import math
from dataclasses import dataclass
from typing import List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


# -----------------------------
# Utils
# -----------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Supports [..., D]
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


def block_attn_res(
    blocks: List[torch.Tensor],
    partial_block: Optional[torch.Tensor],
    proj: nn.Linear,
    norm: RMSNorm,
) -> torch.Tensor:
    """
    Inter-block attention over completed block reps + optional intra-block partial sum.

    Args:
        blocks:
            List of [B, T, D], typically containing:
            - token embedding b0
            - completed block representations b1, b2, ...
        partial_block:
            [B, T, D] or None.
            If not None, this is the current intra-block partial sum.
        proj:
            nn.Linear(D, 1, bias=False), whose weight is the learned pseudo-query.
        norm:
            RMSNorm applied to keys.

    Returns:
        h: [B, T, D]
    """
    sources = list(blocks)
    if partial_block is not None:
        sources.append(partial_block)

    if len(sources) == 0:
        raise ValueError("block_attn_res requires at least one source tensor.")

    # [Nsrc, B, T, D]
    V = torch.stack(sources, dim=0)
    K = norm(V)

    # Query is a learned vector of shape [D]
    q = proj.weight.view(-1)  # [D]

    # logits over source dimension
    # [Nsrc, B, T]
    logits = torch.einsum("d,nbtd->nbt", q, K)

    # softmax over sources
    weights = F.softmax(logits.float(), dim=0).to(dtype=V.dtype)

    # weighted sum of values -> [B, T, D]
    h = torch.einsum("nbt,nbtd->btd", weights, V)
    return h


# -----------------------------
# Transformer submodules
# -----------------------------
class CausalSelfAttention(nn.Module):
    def __init__(
        self,
        dim: int,
        n_heads: int,
        attn_dropout: float = 0.0,
        resid_dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()
        if dim % n_heads != 0:
            raise ValueError(f"dim={dim} must be divisible by n_heads={n_heads}")

        self.dim = dim
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.attn_dropout = attn_dropout

        self.qkv = nn.Linear(dim, 3 * dim, bias=bias)
        self.out_proj = nn.Linear(dim, dim, bias=bias)
        self.resid_dropout = nn.Dropout(resid_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        qkv = self.qkv(x)  # [B, T, 3D]
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)  # [B, H, T, Hd]
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # PyTorch fused causal attention
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )  # [B, H, T, Hd]

        y = y.transpose(1, 2).contiguous().view(B, T, D)
        y = self.out_proj(y)
        y = self.resid_dropout(y)
        return y


class SwiGLUMLP(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: int,
        dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=bias)
        self.w2 = nn.Linear(dim, hidden_dim, bias=bias)
        self.out_proj = nn.Linear(hidden_dim, dim, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.silu(self.w1(x)) * self.w2(x)
        x = self.out_proj(x)
        x = self.dropout(x)
        return x


# -----------------------------
# Block AttnRes Transformer block
# -----------------------------
class BlockAttnResTransformerBlock(nn.Module):
    """
    One standard Transformer block:
      pre-attn BlockAttnRes -> attention -> add into partial_block
      pre-mlp  BlockAttnRes -> mlp       -> add into partial_block
    """

    def __init__(
        self,
        dim: int,
        n_heads: int,
        mlp_hidden_dim: int,
        block_size_in_transformer_blocks: int,
        block_index: int,
        attn_dropout: float = 0.0,
        resid_dropout: float = 0.0,
        mlp_dropout: float = 0.0,
        bias: bool = False,
        norm_eps: float = 1e-6,
    ):
        super().__init__()
        self.block_size_in_transformer_blocks = block_size_in_transformer_blocks
        self.block_index = block_index  # 0-based transformer block index

        # Standard prenorms for attention / mlp
        self.attn_norm = RMSNorm(dim, eps=norm_eps)
        self.mlp_norm = RMSNorm(dim, eps=norm_eps)

        self.attn = CausalSelfAttention(
            dim=dim,
            n_heads=n_heads,
            attn_dropout=attn_dropout,
            resid_dropout=resid_dropout,
            bias=bias,
        )
        self.mlp = SwiGLUMLP(
            dim=dim,
            hidden_dim=mlp_hidden_dim,
            dropout=mlp_dropout,
            bias=bias,
        )

        # AttnRes query projections + key norms
        self.attn_res_proj = nn.Linear(dim, 1, bias=False)
        self.attn_res_norm = RMSNorm(dim, eps=norm_eps)

        self.mlp_res_proj = nn.Linear(dim, 1, bias=False)
        self.mlp_res_norm = RMSNorm(dim, eps=norm_eps)

        self.reset_parameters()

    def reset_parameters(self) -> None:
        # Paper says pseudo-query vectors should be initialized to zero
        nn.init.zeros_(self.attn_res_proj.weight)
        nn.init.zeros_(self.mlp_res_proj.weight)

    def _is_new_block_start(self) -> bool:
        return self.block_index % self.block_size_in_transformer_blocks == 0

    def forward(
        self,
        blocks: List[torch.Tensor],
        partial_block: Optional[torch.Tensor],
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:
        """
        Args:
            blocks:
                Completed block representations, and blocks[0] should be token embeddings.
            partial_block:
                Current intra-block partial sum, or None if this is the first transformer block of a new AttnRes block.

        Returns:
            updated blocks, updated partial_block
        """
        # At the beginning of a new AttnRes block, partial_block should be None.
        # Then pre-attn BlockAttnRes attends only over completed blocks (including token embedding).
        # Otherwise it attends over completed blocks + current partial sum.
        h = block_attn_res(blocks, partial_block, self.attn_res_proj, self.attn_res_norm)

        # Self-attention layer
        attn_out = self.attn(self.attn_norm(h))
        partial_block = attn_out if partial_block is None else (partial_block + attn_out)

        # Apply block attnres before MLP
        h = block_attn_res(blocks, partial_block, self.mlp_res_proj, self.mlp_res_norm)

        # MLP layer
        mlp_out = self.mlp(self.mlp_norm(h))
        partial_block = partial_block + mlp_out

        return blocks, partial_block


# -----------------------------
# Full LM
# -----------------------------
@dataclass
class BlockAttnResConfig:
    vocab_size: int
    max_seq_len: int
    dim: int
    n_heads: int
    n_layers: int
    mlp_hidden_dim: int
    block_size_in_transformer_blocks: int = 3
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    mlp_dropout: float = 0.0
    emb_dropout: float = 0.0
    bias: bool = False
    norm_eps: float = 1e-6
    tie_weights: bool = True


class BlockAttnResTransformerLM(nn.Module):
    """
    A minimal causal LM using Block Attention Residuals.

    Notes:
    - We treat each nn.Module block here as one "Transformer block" = Attention + MLP.
    - The paper counts ATTN and MLP separately as 2 layers for block partitioning.
      Here we group by transformer blocks for simplicity.
    - blocks[0] is always the token embedding b0.
    """

    def __init__(self, cfg: BlockAttnResConfig):
        super().__init__()
        self.cfg = cfg

        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.dim)
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.dim)
        self.emb_dropout = nn.Dropout(cfg.emb_dropout)

        self.layers = nn.ModuleList(
            [
                BlockAttnResTransformerBlock(
                    dim=cfg.dim,
                    n_heads=cfg.n_heads,
                    mlp_hidden_dim=cfg.mlp_hidden_dim,
                    block_size_in_transformer_blocks=cfg.block_size_in_transformer_blocks,
                    block_index=i,
                    attn_dropout=cfg.attn_dropout,
                    resid_dropout=cfg.resid_dropout,
                    mlp_dropout=cfg.mlp_dropout,
                    bias=cfg.bias,
                    norm_eps=cfg.norm_eps,
                )
                for i in range(cfg.n_layers)
            ]
        )

        # Final aggregation over all completed blocks + last partial block
        self.final_res_proj = nn.Linear(cfg.dim, 1, bias=False)
        self.final_res_norm = RMSNorm(cfg.dim, eps=cfg.norm_eps)
        nn.init.zeros_(self.final_res_proj.weight)

        self.final_norm = RMSNorm(cfg.dim, eps=cfg.norm_eps)
        self.lm_head = nn.Linear(cfg.dim, cfg.vocab_size, bias=False)

        if cfg.tie_weights:
            self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            # Keep zero init for pseudo-query projections
            if module is self.final_res_proj:
                return
            if any(
                module is lyr.attn_res_proj or module is lyr.mlp_res_proj
                for lyr in self.layers
            ):
                return

            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(
        self,
        input_ids: torch.Tensor,
        targets: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            input_ids: [B, T]
            targets:   [B, T] or None

        Returns:
            logits: [B, T, vocab_size]
            loss: optional scalar
        """
        B, T = input_ids.shape
        if T > self.cfg.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds max_seq_len {self.cfg.max_seq_len}")

        pos = torch.arange(0, T, device=input_ids.device)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)[None, :, :]
        x = self.emb_dropout(x)

        # blocks always starts with token embedding b0
        blocks: List[torch.Tensor] = [x]
        partial_block: Optional[torch.Tensor] = None

        for i, layer in enumerate(self.layers):
            # If this transformer block starts a new AttnRes block,
            # flush previous partial_block into completed blocks first.
            if i > 0 and i % self.cfg.block_size_in_transformer_blocks == 0:
                if partial_block is None:
                    raise RuntimeError("Expected partial_block to exist at block boundary.")
                blocks.append(partial_block)
                partial_block = None

            blocks, partial_block = layer(blocks, partial_block)

        # Flush final partial block as the last block representation
        if partial_block is not None:
            blocks.append(partial_block)

        # Final output layer aggregates all block representations
        x = block_attn_res(
            blocks=blocks,
            partial_block=None,  # already flushed
            proj=self.final_res_proj,
            norm=self.final_res_norm,
        )
        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )

        return logits, loss



 

In [ ]:
cfg = BlockAttnResConfig(
    vocab_size=32000,
    max_seq_len=1024,
    dim=512,
    n_heads=8,
    n_layers=12,
    mlp_hidden_dim=1536,
    block_size_in_transformer_blocks=3,  # about 4 AttnRes blocks for 12 TF blocks
    attn_dropout=0.0,
    resid_dropout=0.0,
    mlp_dropout=0.0,
    emb_dropout=0.0,
    tie_weights=True,
)

model = BlockAttnResTransformerLM(cfg)
x = torch.randint(0, cfg.vocab_size, (2, 128))
logits, loss = model(x, x)
print("logits:", logits.shape)  # [2, 128, vocab_size]
print("loss:", None if loss is None else loss.item())

In [ ]:
x.shape

In [ ]:
# block_attn_res_encoder.py
from __future__ import annotations

import math
from typing import Optional, Tuple, Dict, Any, List

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """
    RMSNorm over the last dimension.

    Input shape: [..., D]
    Output shape: [..., D]
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = (x_float * rms).to(dtype)
        return y * self.weight.to(dtype)


class BlockAttnResOp(nn.Module):
    """
    Block Attention Residual operation.

    Sources:
        completed blocks: List[[B, T, D]]
        partial_block:   Optional[[B, T, D]]

    Returns:
        h: [B, T, D]

    This implements:
        V = stack(blocks + [partial_block])
        K = RMSNorm(V)
        logits_nbt = <w, K_nbt>
        alpha = softmax(logits, dim=source_dim)
        h = sum_n alpha_nbt * V_nbtd
    """
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.norm = RMSNorm(d_model, eps=eps)

        # 用 Linear 保持和伪代码中的 proj: Linear 一致。
        # bias=False, out_features=1，即一个 learned pseudo-query w_l in R^D。
        self.proj = nn.Linear(d_model, 1, bias=False)

        # 论文强调 pseudo-query 需要零初始化：
        # 初始时 logits 全为 0，softmax 是均匀权重，训练更稳定。
        nn.init.zeros_(self.proj.weight)

    def forward(
        self,
        blocks: List[torch.Tensor],
        partial_block: Optional[torch.Tensor] = None,
        return_weights: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, torch.Tensor]:
        sources: List[torch.Tensor] = list(blocks)
        if partial_block is not None:
            sources.append(partial_block)

        if len(sources) == 0:
            raise ValueError("BlockAttnResOp needs at least one source tensor.")

        # 只有一个 source 时，attention 退化为直接返回该 source。
        if len(sources) == 1:
            h = sources[0]
            if return_weights:
                weights = torch.ones(
                    1, h.shape[0], h.shape[1],
                    device=h.device,
                    dtype=h.dtype,
                )
                return h, weights
            return h

        first_shape = sources[0].shape
        for i, s in enumerate(sources):
            if s.shape != first_shape:
                raise ValueError(
                    f"All BlockAttnRes sources must have the same shape. "
                    f"sources[0]={first_shape}, sources[{i}]={s.shape}"
                )

        # V: [S, B, T, D], where S = number of depth sources.
        V = torch.stack(sources, dim=0)

        # K: [S, B, T, D]
        K = self.norm(V)

        # q: [D]
        q = self.proj.weight.squeeze(0).to(K.dtype)

        # logits: [S, B, T]
        logits = torch.einsum("d,sbtd->sbt", q, K)

        # softmax over depth/source dimension.
        # 用 fp32 做 softmax，混合精度下更稳。
        weights = torch.softmax(logits.float(), dim=0).to(V.dtype)

        # h: [B, T, D]
        h = torch.einsum("sbt,sbtd->btd", weights, V)

        if return_weights:
            return h, weights
        return h


class MultiHeadSelfAttention(nn.Module):
    """
    Self-attention for input sequence features [B, T, D].

    This module does NOT add residual connection internally.
    BlockAttnResEncoder will handle residual aggregation.
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        dropout: float = 0.0,
        bias: bool = False,
        causal: bool = False,
    ):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(f"d_model={d_model} must be divisible by n_heads={n_heads}.")

        self.d_model = d_model
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5
        self.causal = causal

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=bias)
        self.out_proj = nn.Linear(d_model, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Args:
            x:
                [B, T, D]
            attn_mask:
                Optional mask over query-key positions.
                Supported shapes:
                    [T, T], [B, T, T], [B, H, T, T]
                If bool: True means "masked / not allowed".
                If float: additive mask, usually 0 for allowed and -inf for masked.
            key_padding_mask:
                Optional bool tensor [B, T].
                True means the key position is padding and should be masked.

        Returns:
            [B, T, D]
        """
        B, T, D = x.shape

        qkv = self.qkv(x)
        qkv = qkv.view(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, H, T, Hd]
        q, k, v = qkv[0], qkv[1], qkv[2]

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  # [B, H, T, T]

        neg_large = -torch.finfo(scores.dtype).max

        if self.causal:
            causal_mask = torch.ones(T, T, device=x.device, dtype=torch.bool).triu(1)
            scores = scores.masked_fill(causal_mask.view(1, 1, T, T), neg_large)

        if attn_mask is not None:
            attn_mask = attn_mask.to(device=x.device)

            if attn_mask.ndim == 2:
                attn_mask = attn_mask.view(1, 1, T, T)
            elif attn_mask.ndim == 3:
                attn_mask = attn_mask.unsqueeze(1)  # [B, 1, T, T]
            elif attn_mask.ndim == 4:
                pass
            else:
                raise ValueError(
                    "attn_mask must have shape [T,T], [B,T,T], or [B,H,T,T]."
                )

            if attn_mask.dtype == torch.bool:
                scores = scores.masked_fill(attn_mask, neg_large)
            else:
                scores = scores + attn_mask.to(dtype=scores.dtype)

        if key_padding_mask is not None:
            if key_padding_mask.shape != (B, T):
                raise ValueError(
                    f"key_padding_mask must have shape [B,T] = {(B, T)}, "
                    f"got {key_padding_mask.shape}."
                )
            key_padding_mask = key_padding_mask.to(device=x.device, dtype=torch.bool)
            scores = scores.masked_fill(key_padding_mask.view(B, 1, 1, T), neg_large)

        attn = torch.softmax(scores.float(), dim=-1).to(scores.dtype)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v)  # [B, H, T, Hd]
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        out = self.out_proj(out)
        return out


class SwiGLUFeedForward(nn.Module):
    """
    MLP / FFN for [B, T, D].

    This module does NOT add residual connection internally.
    """
    def __init__(
        self,
        d_model: int,
        hidden_dim: Optional[int] = None,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()
        if hidden_dim is None:
            hidden_dim = int(mlp_ratio * d_model)

        self.w12 = nn.Linear(d_model, 2 * hidden_dim, bias=bias)
        self.w3 = nn.Linear(hidden_dim, d_model, bias=bias)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1, x2 = self.w12(x).chunk(2, dim=-1)
        x = F.silu(x1) * x2
        x = self.dropout(x)
        x = self.w3(x)
        return x


class BlockAttnResTransformerLayer(nn.Module):
    """
    One Transformer block containing:
        pre-attention BlockAttnRes + RMSNorm + SelfAttention
        pre-MLP       BlockAttnRes + RMSNorm + MLP

    Important:
        This layer itself does not decide block boundaries.
        BlockAttnResEncoder controls partial_block and completed blocks.
    """
    def __init__(
        self,
        d_model: int,
        n_heads: int,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        bias: bool = False,
        causal: bool = False,
        rms_eps: float = 1e-6,
    ):
        super().__init__()

        self.attn_res = BlockAttnResOp(d_model, eps=rms_eps)
        self.mlp_res = BlockAttnResOp(d_model, eps=rms_eps)

        self.attn_norm = RMSNorm(d_model, eps=rms_eps)
        self.mlp_norm = RMSNorm(d_model, eps=rms_eps)

        self.attn = MultiHeadSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            bias=bias,
            causal=causal,
        )

        self.mlp = SwiGLUFeedForward(
            d_model=d_model,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            bias=bias,
        )


class BlockAttnResEncoder(nn.Module):
    """
    Complete Block AttnRes encoder for sequence features [B, T, D].

    No input_ids.
    No token embedding.
    Input x itself is treated as b0, i.e. the initial representation.

    Args:
        d_model:
            Feature dimension D.
        n_layers:
            Number of Transformer blocks. Each block has Attention + MLP,
            so there are 2 * n_layers residual sublayers.
        n_heads:
            Attention heads.
        num_attnres_blocks:
            Target number of Block AttnRes blocks, excluding b0.
            The actual number can be slightly smaller/larger depending on n_layers.
        attnres_block_size:
            Optional explicit block size counted in sublayers.
            Example: 6 means 3 Transformer blocks per AttnRes block.
            Must be even if you want boundaries only after MLP.
        causal:
            True for autoregressive / causal sequence modeling.
            False for bidirectional encoding over sequence data.
    """
    def __init__(
        self,
        d_model: int,
        n_layers: int,
        n_heads: int,
        num_attnres_blocks: int = 8,
        attnres_block_size: Optional[int] = None,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        bias: bool = False,
        causal: bool = False,
        rms_eps: float = 1e-6,
        final_norm: bool = True,
    ):
        super().__init__()

        if n_layers <= 0:
            raise ValueError("n_layers must be positive.")
        if num_attnres_blocks <= 0:
            raise ValueError("num_attnres_blocks must be positive.")

        self.d_model = d_model
        self.n_layers = n_layers

        # block_size counts Attention + MLP sublayers.
        # To match the usual Transformer-block boundary, make it even by default:
        # layers_per_attnres_block Transformer blocks -> 2 * that many sublayers.
        if attnres_block_size is None:
            layers_per_attnres_block = math.ceil(n_layers / num_attnres_blocks)
            attnres_block_size = 2 * layers_per_attnres_block

        if attnres_block_size <= 0:
            raise ValueError("attnres_block_size must be positive.")
        if attnres_block_size % 2 != 0:
            raise ValueError(
                "attnres_block_size counts Attention + MLP sublayers. "
                "Use an even value if block boundaries should align after MLP."
            )

        self.attnres_block_size = attnres_block_size

        self.layers = nn.ModuleList([
            BlockAttnResTransformerLayer(
                d_model=d_model,
                n_heads=n_heads,
                mlp_ratio=mlp_ratio,
                dropout=dropout,
                bias=bias,
                causal=causal,
                rms_eps=rms_eps,
            )
            for _ in range(n_layers)
        ])

        # Final output aggregation over all block representations.
        self.final_res = BlockAttnResOp(d_model, eps=rms_eps)
        self.final_norm = RMSNorm(d_model, eps=rms_eps) if final_norm else nn.Identity()

    def _maybe_close_block(
        self,
        blocks: List[torch.Tensor],
        partial_block: Optional[torch.Tensor],
        sublayer_idx: int,
    ) -> Optional[torch.Tensor]:
        """
        If current AttnRes block reaches boundary, move partial_block into blocks.
        """
        if partial_block is not None and sublayer_idx % self.attnres_block_size == 0:
            blocks.append(partial_block)
            return None
        return partial_block

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
        return_state: bool = False,
        return_attnres_weights: bool = False,
    ) -> torch.Tensor | Tuple[torch.Tensor, Dict[str, Any]]:
        """
        Args:
            x:
                Input sequence features [B, T, D].
            attn_mask:
                Optional self-attention mask.
            key_padding_mask:
                Optional bool mask [B, T], True means padding.
            return_state:
                Whether to return blocks / partial_block info.
            return_attnres_weights:
                Whether to return BlockAttnRes source weights for inspection.

        Returns:
            y:
                [B, T, D]
            optionally:
                info dict
        """
        if x.ndim != 3:
            raise ValueError(f"Expected x with shape [B,T,D], got {x.shape}.")
        if x.shape[-1] != self.d_model:
            raise ValueError(
                f"Expected last dim D={self.d_model}, got {x.shape[-1]}."
            )

        # b0 = input sequence representation.
        # 注意：不要同时把 x 放进 blocks 又作为 partial_block，
        # 否则第一层会重复 attend 到同一个输入。
        blocks: List[torch.Tensor] = [x]
        partial_block: Optional[torch.Tensor] = None

        sublayer_idx = 0
        attnres_weights: Dict[str, torch.Tensor] = {}

        for layer_idx, layer in enumerate(self.layers):
            # -------- pre-attention Block AttnRes --------
            if return_attnres_weights:
                h, w = layer.attn_res(
                    blocks,
                    partial_block,
                    return_weights=True,
                )
                attnres_weights[f"layer_{layer_idx}.pre_attn"] = w.detach()
            else:
                h = layer.attn_res(blocks, partial_block)

            # Self-attention sublayer output f_l(h_l)
            attn_out = layer.attn(
                layer.attn_norm(h),
                attn_mask=attn_mask,
                key_padding_mask=key_padding_mask,
            )

            # Intra-block standard residual accumulation:
            # b_n^i = b_n^{i-1} + f_i(h_i)
            partial_block = attn_out if partial_block is None else partial_block + attn_out

            sublayer_idx += 1
            partial_block = self._maybe_close_block(
                blocks,
                partial_block,
                sublayer_idx,
            )

            # -------- pre-MLP Block AttnRes --------
            if return_attnres_weights:
                h, w = layer.mlp_res(
                    blocks,
                    partial_block,
                    return_weights=True,
                )
                attnres_weights[f"layer_{layer_idx}.pre_mlp"] = w.detach()
            else:
                h = layer.mlp_res(blocks, partial_block)

            # MLP sublayer output f_l(h_l)
            mlp_out = layer.mlp(layer.mlp_norm(h))

            partial_block = mlp_out if partial_block is None else partial_block + mlp_out

            sublayer_idx += 1
            partial_block = self._maybe_close_block(
                blocks,
                partial_block,
                sublayer_idx,
            )

        # Final output aggregation over completed blocks + last partial block.
        if return_attnres_weights:
            y, w = self.final_res(blocks, partial_block, return_weights=True)
            attnres_weights["final"] = w.detach()
        else:
            y = self.final_res(blocks, partial_block)

        y = self.final_norm(y)

        if return_state or return_attnres_weights:
            info: Dict[str, Any] = {}
            if return_state:
                info["blocks"] = blocks
                info["partial_block"] = partial_block
                info["num_completed_blocks_including_b0"] = len(blocks)
                info["has_partial_block"] = partial_block is not None
                info["attnres_block_size"] = self.attnres_block_size
            if return_attnres_weights:
                info["attnres_weights"] = attnres_weights
            return y, info

        return y




In [ ]:
torch.manual_seed(0)

B, T, D = 2, 128, 256
x = torch.randn(B, T, D)

model = BlockAttnResEncoder(
    d_model=D,
    n_layers=12,
    n_heads=8,
    num_attnres_blocks=8,
    mlp_ratio=4.0,
    dropout=0.1,
    causal=False,          # 普通序列编码用 False；自回归建模用 True
    final_norm=True,
)

y, info = model(
    x,
    return_state=True,
    return_attnres_weights=True,
)

print("input :", x.shape)
print("output:", y.shape)
print("completed blocks including b0:", info["num_completed_blocks_including_b0"])
print("has partial block:", info["has_partial_block"])

loss = y.pow(2).mean()
loss.backward()
print("backward ok")

In [ ]:
# block_attn_res_transformer.py

import math
from typing import Optional, List, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., D]
        x_float = x.float()
        rms = x_float.pow(2).mean(dim=-1, keepdim=True)
        x_norm = x_float * torch.rsqrt(rms + self.eps)
        return x_norm.to(dtype=x.dtype) * self.weight.to(dtype=x.dtype)


def _maybe_padding_mask(
    mask: Optional[torch.Tensor],
    batch: int,
    seq_len: int,
) -> Optional[torch.Tensor]:
    """
    Return [B, L] bool padding mask if mask unambiguously means token validity.

    Convention:
      True / 1 = valid token
      False / 0 = padding token
    """
    if mask is None:
        return None

    if mask.dim() == 2 and tuple(mask.shape) == (batch, seq_len):
        return mask if mask.dtype == torch.bool else (mask > 0)

    return None


def _prepare_sdpa_mask(
    mask: Optional[torch.Tensor],
    *,
    batch: int,
    q_len: int,
    k_len: int,
    device: torch.device,
    dtype: torch.dtype,
    causal: bool,
) -> Tuple[Optional[torch.Tensor], bool]:
    """
    Prepare mask for torch.nn.functional.scaled_dot_product_attention.

    Supported mask forms:

      1. None

      2. [B, L] bool / 0-1 padding mask
         True / 1 = valid token
         False / 0 = padding token

      3. [L, L] bool pairwise attention mask
         True = can attend
         False = blocked

      4. [B, L, L] bool pairwise attention mask
         True = can attend
         False = blocked

      5. [B, 1|H, L, L] bool pairwise attention mask
         True = can attend
         False = blocked

      6. Float masks of rank 2/3/4
         Treated as additive attention bias.
         Usually 0 for keep, -inf or a large negative number for blocked.

    Note:
      PyTorch SDPA bool mask convention is:
        True means the element should take part in attention.
    """
    if mask is None:
        # Let SDPA use its native causal fast path when no external mask is given.
        return None, bool(causal)

    if mask.device != device:
        mask = mask.to(device)

    # Common case: [B, S] padding mask.
    # bool True or numeric >0 means valid key token.
    if mask.dim() == 2 and tuple(mask.shape) == (batch, k_len):
        key_keep = mask if mask.dtype == torch.bool else (mask > 0)
        key_keep = key_keep.to(device=device).clone()

        # Avoid all-False rows, which can create NaNs when a sample is entirely padding.
        # The model input is zeroed for padding positions outside this function,
        # so allowing one dummy key for all-pad samples is safe.
        all_pad = ~key_keep.any(dim=1)
        if all_pad.any():
            key_keep[all_pad, 0] = True

        # [B, 1, 1, S], broadcast over heads and query positions.
        attn_mask = key_keep[:, None, None, :]

        if causal:
            causal_keep = torch.ones(
                (q_len, k_len),
                dtype=torch.bool,
                device=device,
            ).tril()
            # [B, 1, Q, S]
            attn_mask = attn_mask & causal_keep[None, None, :, :]

        return attn_mask, False

    # Bool pairwise mask.
    if mask.dtype == torch.bool:
        attn_mask = mask.to(device=device)

        if attn_mask.dim() == 2:
            if tuple(attn_mask.shape) != (q_len, k_len):
                raise ValueError(
                    f"2D bool attention mask must be [L, S]={q_len, k_len} "
                    f"or [B, S]={batch, k_len}, got {tuple(attn_mask.shape)}"
                )
            attn_mask = attn_mask[None, None, :, :]  # [1, 1, Q, S]

        elif attn_mask.dim() == 3:
            if tuple(attn_mask.shape) != (batch, q_len, k_len):
                raise ValueError(
                    f"3D bool attention mask must be [B, L, S]={batch, q_len, k_len}, "
                    f"got {tuple(attn_mask.shape)}"
                )
            attn_mask = attn_mask[:, None, :, :]  # [B, 1, Q, S]

        elif attn_mask.dim() == 4:
            # Expected [B or 1, H or 1, Q, S].
            # SDPA will broadcast where possible.
            if attn_mask.shape[-2:] != (q_len, k_len):
                raise ValueError(
                    f"4D bool attention mask last dims must be [L, S]={q_len, k_len}, "
                    f"got {tuple(attn_mask.shape)}"
                )

        else:
            raise ValueError(f"Unsupported bool attention mask rank: {attn_mask.dim()}")

        if causal:
            causal_keep = torch.ones(
                (q_len, k_len),
                dtype=torch.bool,
                device=device,
            ).tril()
            attn_mask = attn_mask & causal_keep[None, None, :, :]

        return attn_mask, False

    # Float additive mask.
    # Example:
    #   0 for keep
    #   -inf or a large negative number for blocked
    attn_bias = mask.to(device=device, dtype=dtype)

    if attn_bias.dim() == 2:
        if tuple(attn_bias.shape) != (q_len, k_len):
            raise ValueError(
                f"2D float attention mask must be additive [L, S]={q_len, k_len}. "
                f"[B, S] 0/1 masks are handled as padding masks above. "
                f"Got {tuple(attn_bias.shape)}"
            )
        attn_bias = attn_bias[None, None, :, :]  # [1, 1, Q, S]

    elif attn_bias.dim() == 3:
        if tuple(attn_bias.shape) != (batch, q_len, k_len):
            raise ValueError(
                f"3D float attention mask must be [B, L, S]={batch, q_len, k_len}, "
                f"got {tuple(attn_bias.shape)}"
            )
        attn_bias = attn_bias[:, None, :, :]  # [B, 1, Q, S]

    elif attn_bias.dim() == 4:
        if attn_bias.shape[-2:] != (q_len, k_len):
            raise ValueError(
                f"4D float attention mask last dims must be [L, S]={q_len, k_len}, "
                f"got {tuple(attn_bias.shape)}"
            )

    else:
        raise ValueError(f"Unsupported float attention mask rank: {attn_bias.dim()}")

    if causal:
        causal_keep = torch.ones(
            (q_len, k_len),
            dtype=torch.bool,
            device=device,
        ).tril()

        neg = torch.finfo(dtype).min
        causal_bias = torch.zeros(
            (q_len, k_len),
            device=device,
            dtype=dtype,
        ).masked_fill(~causal_keep, neg)

        attn_bias = attn_bias + causal_bias[None, None, :, :]

    return attn_bias, False


class MultiHeadSelfAttention(nn.Module):
    """
    Self-attention for continuous sequence input [B, L, D].

    Uses torch.nn.functional.scaled_dot_product_attention.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        head_dim: Optional[int] = None,
        attn_dropout: float = 0.0,
        proj_dropout: float = 0.0,
        bias: bool = False,
        causal: bool = False,
    ):
        super().__init__()

        if head_dim is None:
            if dim % num_heads != 0:
                raise ValueError(
                    f"dim={dim} must be divisible by num_heads={num_heads} "
                    f"when head_dim is None"
                )
            head_dim = dim // num_heads

        self.dim = dim
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.inner_dim = num_heads * head_dim
        self.attn_dropout = float(attn_dropout)
        self.causal = bool(causal)

        self.qkv = nn.Linear(dim, 3 * self.inner_dim, bias=bias)
        self.out_proj = nn.Linear(self.inner_dim, dim, bias=bias)
        self.out_dropout = nn.Dropout(proj_dropout)

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        causal: Optional[bool] = None,
    ) -> torch.Tensor:
        """
        Args:
            x:
                [B, L, D]
            mask:
                See _prepare_sdpa_mask.
            causal:
                None means using module default.
        """
        if x.dim() != 3:
            raise ValueError(f"x must be [B, L, D], got {tuple(x.shape)}")

        B, L, _ = x.shape
        causal = self.causal if causal is None else bool(causal)

        qkv = self.qkv(x)  # [B, L, 3 * H * Dh]
        qkv = qkv.view(B, L, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4).contiguous()
        q, k, v = qkv.unbind(dim=0)  # each: [B, H, L, Dh]

        attn_mask, use_is_causal = _prepare_sdpa_mask(
            mask,
            batch=B,
            q_len=L,
            k_len=L,
            device=x.device,
            dtype=q.dtype,
            causal=causal,
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=use_is_causal,
        )  # [B, H, L, Dh]

        y = y.transpose(1, 2).contiguous().view(B, L, self.inner_dim)
        y = self.out_proj(y)
        y = self.out_dropout(y)
        return y


class FeedForward(nn.Module):
    def __init__(
        self,
        dim: int,
        hidden_dim: Optional[int] = None,
        mlp_ratio: float = 4.0,
        dropout: float = 0.0,
        bias: bool = False,
    ):
        super().__init__()

        hidden_dim = hidden_dim or int(dim * mlp_ratio)

        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim, bias=bias),
            nn.GELU(approximate="tanh"),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim, bias=bias),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class BlockAttnRes(nn.Module):
    """
    Inter-block residual attention.

    Given previous completed block representations and current intra-block partial sum,
    compute a per-token softmax over the block axis and return a weighted sum.

    Pseudocode equivalent:

        V = torch.stack(blocks + [partial_block])  # [M, B, L, D]
        K = norm(V)
        logits = proj(K).squeeze(-1)               # [M, B, L]
        h = sum(softmax(logits, dim=0) * V)        # [B, L, D]

    where M = len(blocks) + 1.
    """

    def __init__(
        self,
        dim: int,
        eps: float = 1e-6,
        bias: bool = False,
    ):
        super().__init__()
        self.norm = RMSNorm(dim, eps=eps)
        self.proj = nn.Linear(dim, 1, bias=bias)

    def forward(
        self,
        blocks: List[torch.Tensor],
        partial_block: torch.Tensor,
    ) -> torch.Tensor:
        if partial_block is None:
            raise ValueError("partial_block cannot be None when calling BlockAttnRes")

        if len(blocks) == 0:
            V = partial_block.unsqueeze(0)  # [1, B, L, D]
        else:
            # Every tensor must be [B, L, D].
            V = torch.stack([*blocks, partial_block], dim=0)  # [M, B, L, D]

        K = self.norm(V)
        logits = self.proj(K).squeeze(-1)  # [M, B, L]

        # Softmax over block axis.
        weights = torch.softmax(logits.float(), dim=0).to(dtype=V.dtype)

        h = torch.sum(weights[..., None] * V, dim=0)  # [B, L, D]
        return h


class BlockAttnResLayer(nn.Module):
    """
    One Transformer layer with Block AttnRes.

    Forward order:

      1. Apply BlockAttnRes before self-attention.
      2. At block boundary, append old partial_block to blocks and reset partial_block.
      3. Self-attention layer.
      4. Accumulate attention output into current partial_block.
      5. Apply BlockAttnRes before MLP.
      6. MLP layer.
      7. Accumulate MLP output into current partial_block.
    """

    def __init__(
        self,
        dim: int,
        num_heads: int,
        head_dim: Optional[int] = None,
        mlp_ratio: float = 4.0,
        mlp_hidden_dim: Optional[int] = None,
        attn_dropout: float = 0.0,
        dropout: float = 0.0,
        bias: bool = False,
        norm_eps: float = 1e-6,
        causal: bool = False,
    ):
        super().__init__()

        self.attn_res = BlockAttnRes(dim, eps=norm_eps, bias=bias)
        self.mlp_res = BlockAttnRes(dim, eps=norm_eps, bias=bias)

        self.attn_norm = RMSNorm(dim, eps=norm_eps)
        self.mlp_norm = RMSNorm(dim, eps=norm_eps)

        self.attn = MultiHeadSelfAttention(
            dim=dim,
            num_heads=num_heads,
            head_dim=head_dim,
            attn_dropout=attn_dropout,
            proj_dropout=dropout,
            bias=bias,
            causal=causal,
        )

        self.mlp = FeedForward(
            dim=dim,
            hidden_dim=mlp_hidden_dim,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
            bias=bias,
        )

    def forward(
        self,
        blocks: List[torch.Tensor],
        partial_block: torch.Tensor,
        *,
        mask: Optional[torch.Tensor] = None,
        causal: Optional[bool] = None,
        start_new_block: bool = False,
        padding_mask: Optional[torch.Tensor] = None,
    ) -> Tuple[List[torch.Tensor], torch.Tensor]:

        # 1. Apply Block AttnRes before self-attention.
        h = self.attn_res(blocks, partial_block)

        # 2. At a block boundary, the old partial_block becomes a completed block.
        #    The next self-attention output starts the new partial_block.
        if start_new_block:
            blocks = [*blocks, partial_block]
            partial_block = None

        # 3. Self-attention.
        attn_out = self.attn(
            self.attn_norm(h),
            mask=mask,
            causal=causal,
        )

        if padding_mask is not None:
            attn_out = attn_out * padding_mask[:, :, None].to(dtype=attn_out.dtype)

        # 4. Accumulate attention output.
        partial_block = attn_out if partial_block is None else partial_block + attn_out

        if padding_mask is not None:
            partial_block = partial_block * padding_mask[:, :, None].to(
                dtype=partial_block.dtype
            )

        # 5. Apply Block AttnRes before MLP.
        h = self.mlp_res(blocks, partial_block)

        # 6. MLP.
        mlp_out = self.mlp(self.mlp_norm(h))

        if padding_mask is not None:
            mlp_out = mlp_out * padding_mask[:, :, None].to(dtype=mlp_out.dtype)

        # 7. Accumulate MLP output.
        partial_block = partial_block + mlp_out

        if padding_mask is not None:
            partial_block = partial_block * padding_mask[:, :, None].to(
                dtype=partial_block.dtype
            )

        return blocks, partial_block


class BlockAttnResTransformer(nn.Module):
    """
    Block AttnRes Transformer for continuous sequence input [B, L, D_in].

    Args:
        input_dim:
            Last dimension of input sequence. If None, equals dim.

        dim:
            Model width.

        depth:
            Number of Transformer layers.

        num_heads:
            Number of attention heads.

        block_size:
            Number of sublayers per block.
            Attention and MLP each count as 1 sublayer.
            Therefore:
                layers_per_block = block_size // 2

            Example:
                block_size = 4 means every 2 Transformer layers form one block.

            If block_size is None, it is inferred from num_blocks.

        num_blocks:
            Approximate target block count used only when block_size is None.

        include_input_block:
            True means initialize completed blocks with the input representation,
            matching the pseudocode comment:
                "blocks already include token embedding"

            For continuous data, this initial representation is input_proj(x).

        final_mix:
            True means apply one final BlockAttnRes over completed blocks + partial block.
            False means return the final partial_block after final_norm.
    """

    def __init__(
        self,
        *,
        dim: int,
        depth: int,
        num_heads: int,
        input_dim: Optional[int] = None,
        output_dim: Optional[int] = None,
        head_dim: Optional[int] = None,
        mlp_ratio: float = 4.0,
        mlp_hidden_dim: Optional[int] = None,
        block_size: Optional[int] = None,
        num_blocks: int = 8,
        attn_dropout: float = 0.0,
        dropout: float = 0.0,
        bias: bool = False,
        norm_eps: float = 1e-6,
        causal: bool = False,
        include_input_block: bool = True,
        final_mix: bool = True,
    ):
        super().__init__()

        if depth < 1:
            raise ValueError("depth must be >= 1")

        if block_size is not None and block_size < 2:
            raise ValueError(
                "block_size must be >= 2 because each Transformer layer has attention + MLP"
            )

        if block_size is not None and block_size % 2 != 0:
            raise ValueError(
                "block_size must be even: attention and MLP each count as one sublayer"
            )

        input_dim = input_dim or dim
        output_dim = output_dim or dim

        self.dim = dim
        self.depth = depth
        self.causal = bool(causal)
        self.include_input_block = bool(include_input_block)
        self.final_mix_enabled = bool(final_mix)

        if block_size is None:
            # Approximate "N blocks" by grouping full Transformer layers.
            layers_per_block = max(1, math.ceil(depth / max(1, num_blocks)))
            block_size = 2 * layers_per_block
        else:
            layers_per_block = max(1, block_size // 2)

        self.block_size = block_size
        self.layers_per_block = layers_per_block

        self.input_proj = (
            nn.Identity()
            if input_dim == dim
            else nn.Linear(input_dim, dim, bias=bias)
        )

        self.layers = nn.ModuleList(
            [
                BlockAttnResLayer(
                    dim=dim,
                    num_heads=num_heads,
                    head_dim=head_dim,
                    mlp_ratio=mlp_ratio,
                    mlp_hidden_dim=mlp_hidden_dim,
                    attn_dropout=attn_dropout,
                    dropout=dropout,
                    bias=bias,
                    norm_eps=norm_eps,
                    causal=causal,
                )
                for _ in range(depth)
            ]
        )

        self.final_attn_res = BlockAttnRes(dim, eps=norm_eps, bias=bias)
        self.final_norm = RMSNorm(dim, eps=norm_eps)

        self.output_proj = (
            nn.Identity()
            if output_dim == dim
            else nn.Linear(dim, output_dim, bias=bias)
        )

    def forward(
        self,
        x: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        *,
        causal: Optional[bool] = None,
        return_blocks: bool = False,
    ):
        """
        Args:
            x:
                Continuous sequence input, shape [B, L, D_in].

            mask:
                Supported forms:

                1. [B, L] bool / 0-1 padding mask
                   True / 1 = valid token
                   False / 0 = padding token

                2. [L, L] bool attention mask
                   True = can attend

                3. [B, L, L] bool attention mask
                   True = can attend

                4. [B, 1|H, L, L] bool attention mask
                   True = can attend

                5. Float masks of rank 2/3/4
                   Treated as additive attention bias.
                   Usually 0 for keep, -inf for blocked.

                Note:
                   A float [B, L] 0/1 mask is treated as padding mask.

            causal:
                None means using module default.
                True applies causal self-attention.
                When mask is also given, causal mask and user mask are combined.

            return_blocks:
                True returns:
                    output, aux_dict

                aux_dict contains:
                    blocks
                    partial_block
                    block_size
                    layers_per_block
        """
        if x.dim() != 3:
            raise ValueError(f"x must be [B, L, D], got {tuple(x.shape)}")

        B, L, _ = x.shape
        causal = self.causal if causal is None else bool(causal)

        x = self.input_proj(x)

        padding_mask = _maybe_padding_mask(mask, B, L)
        if padding_mask is not None:
            padding_mask = padding_mask.to(device=x.device)
            x = x * padding_mask[:, :, None].to(dtype=x.dtype)

        # For continuous data, this is analogous to putting token embedding
        # into the completed block list.
        blocks: List[torch.Tensor] = [x] if self.include_input_block else []
        partial_block = x

        for layer_idx, layer in enumerate(self.layers):
            # 0-based implementation of block boundary.
            #
            # Every `layers_per_block` Transformer layers form one block.
            # Do not cut before the first layer.
            start_new_block = (
                layer_idx > 0
                and layer_idx % self.layers_per_block == 0
            )

            blocks, partial_block = layer(
                blocks,
                partial_block,
                mask=mask,
                causal=causal,
                start_new_block=start_new_block,
                padding_mask=padding_mask,
            )

        if self.final_mix_enabled:
            hidden = self.final_attn_res(blocks, partial_block)
        else:
            hidden = partial_block

        hidden = self.final_norm(hidden)

        if padding_mask is not None:
            hidden = hidden * padding_mask[:, :, None].to(dtype=hidden.dtype)

        out = self.output_proj(hidden)

        if return_blocks:
            return out, {
                "blocks": blocks,
                "partial_block": partial_block,
                "block_size": self.block_size,
                "layers_per_block": self.layers_per_block,
            }

        return out